In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '6,7'

In [2]:
import pandas
import json

In [3]:
pd = pandas.read_json("data/dyn_topk/ceval/ceval_results.json")

In [10]:
entries = pd[pd['subject'] == 'high_school_chemistry'].iloc[:2]
entries

,id,subject,question,think_part,stop_reason
5331,0,high_school_chemistry,有下列 5 种物质：①$C_2H_4$ 和 $H_2$，②$H_2$ 和 $Cl_2$，③$...,嗯，我现在要解决这个题目，题目是说有五种物质，分别是①C₂H₄和H₂，②H₂和Cl₂，③CH...,stop
5332,1,high_school_chemistry,钙是人体必需的常量元素，成年人每天需要800毫克的钙，下列补钙的途径不正确的是____,嗯，我现在要解决这个关于补钙的问题。题目是说，成年人每天需要800毫克的钙，然后问哪个补钙的...,stop


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [6]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-30B-A3B", device_map='auto', torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2",)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-30B-A3B", device_map='auto')

Loading checkpoint shards:   0%|          | 0/16 [00:00<?, ?it/s]

In [11]:
def apply_template(entry):
    return f'{entry["question"]}<think>{entry["think_part"]}</think>'

seqs = [apply_template(entry) for entry in entries.iloc]
inputs = tokenizer(seqs, return_tensors='pt', padding=True, truncation=True).to(model.device)

inputs

{'input_ids': tensor([[ 18830, 107976,    220,  ...,     33,     92, 151668],
        [106033,  20412, 102314,  ..., 151643, 151643, 151643]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')}

In [12]:
inputs.input_ids.shape

torch.Size([2, 2423])

In [16]:
outputs = model(**inputs)

In [21]:
outputs.logits.shape

torch.Size([2, 2423, 151936])